In [18]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

In [19]:
geo_df = pd.read_csv(
    PROCESSED_DIR
    / "eurosat_geo_clusters.csv"
)

print("Original Shape:")
print(geo_df.shape)

geo_df.head()

Original Shape:
(27000, 7)


,filepath_rgb,filepath_tif,class_name,center_lat,center_lon,class_id,geo_cluster
0,c:\Users\ASUS\dev\projects\satellite-project-c...,c:\Users\ASUS\dev\projects\satellite-project-c...,AnnualCrop,44.035220,28.559055,0,3
1,c:\Users\ASUS\dev\projects\satellite-project-c...,c:\Users\ASUS\dev\projects\satellite-project-c...,AnnualCrop,39.085801,-1.829726,0,5
2,c:\Users\ASUS\dev\projects\satellite-project-c...,c:\Users\ASUS\dev\projects\satellite-project-c...,AnnualCrop,48.977295,4.239720,0,2
3,c:\Users\ASUS\dev\projects\satellite-project-c...,c:\Users\ASUS\dev\projects\satellite-project-c...,AnnualCrop,48.892610,4.089878,0,2
4,c:\Users\ASUS\dev\projects\satellite-project-c...,c:\Users\ASUS\dev\projects\satellite-project-c...,AnnualCrop,51.832851,18.084960,0,6


In [20]:
geo_df = geo_df[
    geo_df["filepath_rgb"]
    .apply(lambda x: Path(x).exists())
].copy()

print("Filtered Shape:")
print(geo_df.shape)

Filtered Shape:
(27000, 7)


In [21]:
geo_df["class_name"].value_counts()

class_name
AnnualCrop              3000
Forest                  3000
HerbaceousVegetation    3000
Residential             3000
SeaLake                 3000
Highway                 2500
PermanentCrop           2500
Industrial              2500
River                   2500
Pasture                 2000
Name: count, dtype: int64

In [22]:
TRAIN_CLUSTERS = [0, 2, 4, 6]
VAL_CLUSTERS   = [3]
TEST_CLUSTERS  = [5]

In [23]:
geo_df["split"] = "unassigned"

geo_df.loc[
    geo_df["geo_cluster"].isin(TRAIN_CLUSTERS),
    "split"
] = "train"

geo_df.loc[
    geo_df["geo_cluster"].isin(VAL_CLUSTERS),
    "split"
] = "val"

geo_df.loc[
    geo_df["geo_cluster"].isin(TEST_CLUSTERS),
    "split"
] = "test"

geo_df["split"].value_counts()

split
train    21568
val       2774
test      2658
Name: count, dtype: int64

In [24]:
for split_name in [
    "train",
    "val",
    "test"
]:

    subset = geo_df[
        geo_df["split"] == split_name
    ]

    print(
        f"{split_name}: {len(subset)}"
    )

train: 21568
val: 2774
test: 2658


In [25]:
class_distribution = pd.crosstab(
    geo_df["split"],
    geo_df["class_name"]
)

class_distribution

class_name,AnnualCrop,Forest,HerbaceousVegetation,Highway,Industrial,Pasture,PermanentCrop,Residential,River,SeaLake
split,,,,,,,,,,
test,275,105,707,399,244,22,465,247,107,87
train,2443,2465,1481,2031,2094,1936,1731,2382,2323,2682
val,282,430,812,70,162,42,304,371,70,231


In [26]:
cluster_class_table = pd.crosstab(
    geo_df["geo_cluster"],
    geo_df["class_name"]
)

cluster_class_table

class_name,AnnualCrop,Forest,HerbaceousVegetation,Highway,Industrial,Pasture,PermanentCrop,Residential,River,SeaLake
geo_cluster,,,,,,,,,,
0,629,727,323,364,444,112,480,411,577,475
2,1470,1275,234,989,1043,391,1201,1246,1280,697
3,282,430,812,70,162,42,304,371,70,231
4,67,19,903,299,380,1356,50,397,156,219
5,275,105,707,399,244,22,465,247,107,87
6,277,444,21,379,227,77,0,328,310,1291


In [27]:
cluster_class_table.to_csv(
    PROCESSED_DIR /
    "cluster_class_distribution.csv"
)

In [28]:
cluster_summary = pd.DataFrame({
    "cluster_size":
        geo_df["geo_cluster"]
        .value_counts()
        .sort_index()
})

cluster_summary

,cluster_size
geo_cluster,
0,4542
2,9826
3,2774
4,3846
5,2658
6,3354


In [29]:
coverage = (
    cluster_class_table > 0
).sum(axis=1)

coverage

geo_cluster
0    10
2    10
3    10
4    10
5    10
6     9
dtype: int64

In [30]:
class_distribution_pct = (
    class_distribution
    .div(
        class_distribution.sum(axis=1),
        axis=0
    )
)

class_distribution_pct.round(3)

class_name,AnnualCrop,Forest,HerbaceousVegetation,Highway,Industrial,Pasture,PermanentCrop,Residential,River,SeaLake
split,,,,,,,,,,
test,0.103,0.040,0.266,0.150,0.092,0.008,0.175,0.093,0.040,0.033
train,0.113,0.114,0.069,0.094,0.097,0.090,0.080,0.110,0.108,0.124
val,0.102,0.155,0.293,0.025,0.058,0.015,0.110,0.134,0.025,0.083


In [31]:
train_df = geo_df[
    geo_df["split"] == "train"
].copy()

val_df = geo_df[
    geo_df["split"] == "val"
].copy()

test_df = geo_df[
    geo_df["split"] == "test"
].copy()

In [32]:
train_df.to_csv(
    PROCESSED_DIR
    / "train_spatial.csv",
    index=False
)

val_df.to_csv(
    PROCESSED_DIR
    / "val_spatial.csv",
    index=False
)

test_df.to_csv(
    PROCESSED_DIR
    / "test_spatial.csv",
    index=False
)

print("Spatial split files saved.")

Spatial split files saved.


In [33]:
for file in [
    "train_spatial.csv",
    "val_spatial.csv",
    "test_spatial.csv"
]:

    df = pd.read_csv(
        PROCESSED_DIR / file
    )

    print(
        f"{file}: {df.shape}"
    )

train_spatial.csv: (21568, 8)
val_spatial.csv: (2774, 8)
test_spatial.csv: (2658, 8)


In [34]:
print(
    len(train_df)
    + len(val_df)
    + len(test_df)
)

27000
